In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
!pip install transformers

In [26]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import shutil
import string
from sklearn.metrics import classification_report
import tensorflow as tf
from keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, Dropout, Bidirectional, LSTM, Embedding, BatchNormalization,  Reshape, Conv2D, MaxPool2D, concatenate, Flatten, Activation
import torch
import numpy as np
from transformers import BertTokenizer, BertModel,RobertaTokenizer, RobertaModel,AutoTokenizer, AutoModel
import ast

In [27]:
# Verificar si la GPU está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [28]:
archivo_3 = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsGraph/DataProgramsandDescriptions-CatRetroalimentacion5000.xlsx'

train_full = pd.read_excel(archivo_3)

# AST spliter D. Gries form

In [29]:
class WhileLoopFinder(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code.splitlines()
        self.functions_with_while = []
        self.current_function = None

    def visit_FunctionDef(self, node):
        original_current_function = self.current_function
        self.current_function = {
            "function_name": node.name,
            "has_while_loop": False,
            "pre_while_code_lines": [],
            "while_loops": []
        }

        function_start_line = node.lineno - 1
        function_end_line = (node.end_lineno if hasattr(node, 'end_lineno') else len(self.source_code))
        function_lines = self.source_code[function_start_line:function_end_line]

        found_while = False
        for stmt in node.body:
            if isinstance(stmt, ast.While):
                self.current_function["has_while_loop"] = True
                found_while = True
                try:
                    while_condition = ast.unparse(stmt.test).strip()
                    print("OK condition")
                except AttributeError:
                    print("Error al obtener la condición del while")
                try:
                    while_body_code = ast.unparse(ast.Module(body=stmt.body, type_ignores=[])).strip()
                    print("Ok body")
                except AttributeError:
                    print("Error al obtener el código del cuerpo del while")

                self.current_function["while_loops"].append({
                    "condition": while_condition,
                    "body_code": while_body_code
                })
            elif not found_while:
                start_line = stmt.lineno - 1
                end_line = (stmt.end_lineno if hasattr(stmt, 'end_lineno') else stmt.lineno)
                self.current_function["pre_while_code_lines"].extend(self.source_code[start_line:end_line])
            self.generic_visit(stmt)

        if self.current_function["has_while_loop"]:
            last_while_end_line = None
            if self.current_function["while_loops"]:
                last_while_node = next((node for node in reversed(node.body) if isinstance(node, ast.While)), None)
                if last_while_node and hasattr(last_while_node, 'end_lineno'):
                    last_while_end_line = last_while_node.end_lineno
            post_while_code_lines = []
            if last_while_end_line:
                function_indent = len(self.source_code[node.lineno - 1]) - len(self.source_code[node.lineno - 1].lstrip())
                for line in self.source_code[last_while_end_line:function_end_line]:
                    if line.startswith(self.source_code[node.lineno - 1][:function_indent] + "    "):
                        post_while_code_lines.append(line[function_indent + 4:])
                    else:
                        post_while_code_lines.append(line[function_indent:])
            self.current_function["post_while_code_lines"] = "\n".join(post_while_code_lines).strip()
            self.current_function["pre_while_code_lines"] = "\n".join(self.current_function["pre_while_code_lines"]).strip()
            self.functions_with_while.append(self.current_function)

        self.current_function = original_current_function



In [30]:
def DGries_states(solution):
  try:
    # Crear el AST
    tree = ast.parse(solution)
    # Recorrer el AST con nuestro visitante
    finder = WhileLoopFinder(solution)
    finder.visit(tree)
    for func_info in finder.functions_with_while:
      initial_state=func_info["pre_while_code_lines"] if func_info["pre_while_code_lines"] else False
      end_state=""
      transformation_state=""
      for j, wl in enumerate(func_info["while_loops"]):
          end_state=wl["condition"]
          transformation_state=wl["body_code"]

      if not initial_state:
        initial_state=transformation_state
      return initial_state,transformation_state,end_state
  except Exception as error:
    return "Exception:"+str(error),"Exception:"+str(error),"Exception:"+str(error)




# Encoder Description and Code

In [31]:
class Encoder:
  def __init__(self):
    self.is_loadtokenizers=False



  def tokenize_and_generate_embeddings_descriptions(self,description):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input description and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(description, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.beart_model(**tokens)
    # Extract embeddings for all tokens
    desc_embeddings = outputs.last_hidden_state.cpu().numpy()
    return desc_embeddings

  def tokenize_and_generate_embeddings_codes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.codebeart_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.codebeart_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()

    return code_embeddings


  def tokenize_and_generate_embeddings_graphcodes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.graphcodebert_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.graphcodebert_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()
    return code_embeddings

  def load_beart_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    model_name = "bert-base-uncased"
    self.beart_model = BertModel.from_pretrained(model_name)
    self.beart_tokenizer = BertTokenizer.from_pretrained(model_name)
    self.beart_model.to(device)

  def load_graphcodebert_tokenizer(self):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    self.graphcodebert_tokenizer = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model = AutoModel.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model.to(device)


  def load_codebert_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    self.codebeart_tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    self.codebeart_model = RobertaModel.from_pretrained("microsoft/codebert-base")
    self.codebeart_model.to(device)

  def start_tokenizer(self):
    self.load_beart_tokenizer()
    #self.load_codebert_tokenizer()
    self.load_graphcodebert_tokenizer()
    self.is_loadtokenizers=True




# All characteristics

In [32]:
def categoricallabelAll(w):

  if w=="['Initial state']":
    return 0
  if w=="['Final state']":
    return 1
  if w=="['State transformation']":
    return 2
  if w=="['Initial state', 'Final state']":
    return 3
  if w=="['Initial state', 'State transformation']":
    return 4
  if w=="['Final state', 'State transformation']":
    return 5
  if w=="['Initial state', 'Final state', 'State transformation']":
    return 6
  return 7

category=np.array([
    'Initial state',
    'Final state',
    'State transformation',
    'Initial state, Final state',
    'Initial state, State transformation',
    'Final state, State transformation',
    'Initial state, Final state, State transformation'

])

# Load Dataset

In [33]:
train_full.head()

,No.,Problema,Solución,Estado incial,Estado final,Transformación de estado,Etiqueta 1,Etiqueta 2,Realimentación
0,1,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,"result = 1, i = 1",i <= n,"result *= i, i += 1",Correct,['Correct'],NaN
1,2,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,"total = 0, i = 1, n = 100",i <= n,"total += i, i += 1",Correct,['Correct'],NaN
2,3,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,"numbers = [], i = 0, n = 10",i <= n,"print(i), numbers.append(i), i += 1",Correct,['Correct'],NaN
3,4,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,"numbers = [], i = 10, n = 1",i >= n,"print(i), numbers.append(i), i -= 1",Correct,['Correct'],NaN
4,5,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,"i = 2, i = = 0:",i <= num//2,"if num % i == 0:, return False, i += 1",Correct,['Correct'],NaN


In [34]:
train_full.drop(["No.","Realimentación","Estado incial","Estado final","Transformación de estado"],axis=1,inplace=True)
train_full

,Problema,Solución,Etiqueta 1,Etiqueta 2
0,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,Correct,['Correct']
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,Correct,['Correct']
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,Correct,['Correct']
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,Correct,['Correct']
4,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,Correct,['Correct']
...,...,...,...,...
4995,Write a Python function that returns the sum o...,def sum_of_first_five_numbers():\n number =...,Incorrect,"['Initial state', 'Final state']"
4996,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 0\n...,Incorrect,"['Initial state', 'Final state']"
4997,Write a Python function that prints the number...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"
4998,Write a Python function to print the numbers f...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"


In [35]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Solución'].apply(DGries_states).apply(pd.Series)

Streaming output truncated to the last 5000 lines.
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK c

In [36]:
train_full = train_full[train_full['Etiqueta 1']!='Correct'].copy()

In [37]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [38]:
train_full.dropna(inplace=True)

In [39]:
np.unique(train_full['Etiqueta 2'])

array(["['Final state', 'State transformation']", "['Final state']",
       "['Initial state', 'Final state', 'State transformation']",
       "['Initial state', 'Final state']",
       "['Initial state', 'State transformation']", "['Initial state']",
       "['State transformation']"], dtype=object)

In [40]:
encoder=Encoder()
encoder.start_tokenizer()

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [41]:
%%time
problem=train_full['Problema'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

CPU times: user 1min 53s, sys: 1.93 s, total: 1min 55s
Wall time: 1min 52s


In [42]:
problem.shape,startstate.shape,finalstate.shape,transstate.shape

((3459,), (3459,), (3459,), (3459,))

In [43]:
print(startstate.shape)
print(startstate[1262].shape)


(3459,)
(1, 13, 768)


In [44]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((3459, 768), (3459, 768), (3459, 768), (3459, 768))

In [45]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
y=y.to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [46]:
from sklearn.model_selection import train_test_split

Xp_train,Xp_test,Xs_train,Xs_test,Xt_train,Xt_test,Xf_train,Xf_test,y_train,y_test=train_test_split(Xp,Xs,Xt,Xf,y,test_size=0.2,random_state=2023, stratify=y)

In [47]:
Xp_train.shape,Xp_test.shape,Xs_train.shape,Xs_test.shape,Xt_train.shape,Xt_test.shape,Xf_train.shape,Xf_test.shape,y_train.shape,y_test.shape

((2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767,),
 (692,))

# Keras model

In [48]:
3//2

1

In [49]:
from tensorflow.keras import backend as K
import gc
def ANNTC(input,base,pow_initial,num_max_blocks):

  drop_out=0.5
  print("base",base,"pow",pow_initial,"num_max_blocks",num_max_blocks)
  n=num_max_blocks//2
  neurons=int(base**(pow_initial+n+1))
  # Encoder
  x=None
  print("Encoder",n)
  try:
    for i in range(n):
      x=Dense(neurons)(input)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      input=x
      drop_out=0.2
      print("Block ",i,neurons)
      neurons=int(neurons/base)



    #BottleNeck
    x=Dense(neurons)(x)
    x=BatchNormalization()(x)
    x=Activation('relu')(x)
    x=Dropout(0.2)(x)
    print("BottleNeck",neurons)

    # Decoder
    print("Decoder",n)
    for i in range(n):
      neurons=int(neurons*base)
      print("Block ",i,neurons)
      x=Dense(neurons)(x)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      if i==n-1:
        drop_out=0.5
      drop_out=0.2
  except Exception as err:

    K.clear_session()
    gc.collect()
    del x
    print(f"Unexpected {err=}, {type(err)=}")
    raise

  return x


def neural_network(problem,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p, mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem_input, start_input, trass_input,final_input], outputs=output)
  return model

In [50]:
import numpy as np
np.linspace(5,70,14)


array([ 5., 10., 15., 20., 25., 30., 35., 40., 45., 50., 55., 60., 65.,
       70.])

In [51]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from time import time
#tensorflow random state
tf.random.set_seed(2023)

times=[]
max_accuracies=[]
max_val_accuracies=[]
max_val_losses=[]
max_losses=[]
learning_rates=[]


for patience in np.linspace(5,70,14):
  print("Paciencia",patience)
  problem_input=Input((768,))
  start_input=Input((768,))
  trass_input=Input((768,))
  final_input=Input((768,))


  my_model=neural_network(problem_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3)
  my_model.compile(loss='sparse_categorical_crossentropy',
                optimizer=Adam(),
                metrics=['accuracy'])

  es = EarlyStopping(monitor='val_loss', mode='min', patience=patience)

  mc = ModelCheckpoint('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsGraph/early_best_finall_patience{0}.keras'.format(patience), monitor='val_loss', mode='min', save_best_only=True)
  reduce_lr = ReduceLROnPlateau(
      monitor='val_loss',
      factor=0.1,
      patience=patience//2,
      min_lr=1e-9,
      verbose=1
  )

  time1=time()
  history=my_model.fit([Xp_train, Xs_train, Xt_train,Xf_train],y_train,batch_size=100,epochs=1000,
            validation_split=0.2,callbacks=[es, mc,reduce_lr])
  time2=time()

  df_histories=pd.DataFrame(history.history)
  df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsGraph/early_best_finall_{0}.csv'.format(patience))

  times.append(time2-time1)

  max_accuracies.append(max(history.history['accuracy']))
  max_val_accuracies.append(max(history.history['val_accuracy']))
  max_val_losses.append(max(history.history['val_loss']))
  max_losses.append(max(history.history['loss']))
  learning_rates.append(min(history.history['learning_rate']))

  print(f"{patience}")
  print("times=",time2-time1)
  print("max_val_accuracy",max(history.history['val_accuracy']))
  print("max_accuracy",max(history.history['accuracy']))


  print("max_val_loss",max(history.history['val_loss']))
  print("max_loss",max(history.history['loss']))
  print("learning_rate",min(history.history['learning_rate']))





Streaming output truncated to the last 5000 lines.
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9546 - loss: 0.2955 - val_accuracy: 0.9007 - val_loss: 0.3977 - learning_rate: 0.0010
Epoch 45/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9569 - loss: 0.2895 - val_accuracy: 0.8935 - val_loss: 0.4016 - learning_rate: 0.0010
Epoch 46/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9582 - loss: 0.2712 - val_accuracy: 0.8971 - val_loss: 0.3832 - learning_rate: 0.0010
Epoch 47/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9581 - loss: 0.2735 - val_accuracy: 0.8971 - val_loss: 0.3881 - learning_rate: 0.0010
Epoch 48/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9577 - loss: 0.2692 - val_accuracy: 0.8971 - val_loss: 0.3879 - learning_rate: 0.0010
Epoch 49/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9625 - loss: 0.2630 - val_accuracy: 0.9007 - val_loss: 0.3756 - learning_rate: 0.0010
Epoch 50/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 

In [52]:
df_histories=pd.DataFrame({"times":times,"max_accuracies":max_accuracies,"max_val_accuracies":max_val_accuracies,"max_val_losses":max_val_losses,"max_losses":max_losses,"learning_rates":learning_rates})
df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsGraph/early_best_finall.csv')

In [53]:
df_histories

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates
0,59.287467,0.975147,0.909747,1.840268,2.005214,1.000000e-07
1,68.628585,0.990511,0.931408,1.914578,2.010741,1.000000e-05
2,73.777139,0.990059,0.927798,1.864615,2.022250,1.000000e-08
3,90.885048,0.998192,0.936823,1.925896,2.052175,1.000000e-05
4,77.420167,0.996837,0.938628,1.918162,2.032023,1.000000e-06
5,89.939603,0.998192,0.940433,1.926531,2.043819,1.000000e-05
6,85.963678,0.998644,0.945848,1.903825,2.034818,1.000000e-06
7,92.671117,0.998644,0.938628,1.866122,2.037306,1.000000e-05
8,101.977762,0.999548,0.936823,1.940450,2.022266,1.000000e-06
9,98.848299,0.999096,0.940433,1.873892,2.011722,1.000000e-06


In [54]:
from sklearn.metrics import matthews_corrcoef, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

def calculate_mcc_multiclass(y_true, y_pred_probs):
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    # Convertir etiquetas verdaderas one-hot a etiquetas enteras si es necesario
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    return matthews_corrcoef(y_true, y_pred_labels)

def calculate_auc_pr_multiclass(y_true, y_pred_probs, average='macro'):
    n_classes = y_pred_probs.shape[1]
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    auc_pr_list = []
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
        auc_pr_list.append(auc(recall, precision))

    if average == 'macro':
        return np.mean(auc_pr_list)
    elif average == 'weighted':
        class_counts = y_true_bin.sum(axis=0)
        return np.average(auc_pr_list, weights=class_counts)
    else:
        return auc_pr_list


In [55]:
from tensorflow import keras
accuracies_predict=[]
loss_predict=[]
times_predict=[]
mccs_predict=[]
aucpr_predict=[]

for patience in np.linspace(5,70,14):
  model=keras.models.load_model('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsGraph/early_best_finall_patience{0}.keras'.format(patience))
  evaluate=model.evaluate([Xp_test,Xs_test,Xt_test,Xf_test],y_test)

  t1=time()
  y_pred=model.predict([Xp_test,Xs_test,Xt_test,Xf_test])
  t2=time()
  mcc=calculate_mcc_multiclass(y_test, y_pred)
  auc_pr=calculate_auc_pr_multiclass(y_test, y_pred)

  times_predict.append(t2-t1)
  accuracies_predict.append(evaluate[1])
  loss_predict.append(evaluate[0])
  mccs_predict.append(mcc)
  aucpr_predict.append(auc_pr)


  accuracy=evaluate[1]
  loss=evaluate[0]

  print("accuracy",accuracy)
  print("loss",loss)
  print("mcc",mcc)
  print("auc_pr",auc_pr)
  print("time predict",t2-t1)

22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.9054 - loss: 0.3775
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step
accuracy 0.9031791687011719
loss 0.3979676067829132
mcc 0.8706375521236198
auc_pr 0.8710968526225562
time predict 1.6487841606140137


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.9305 - loss: 0.2809
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step
accuracy 0.9219653010368347
loss 0.3130095303058624
mcc 0.8957836641958257
auc_pr 0.8877855433232815
time predict 1.6092088222503662


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.9096 - loss: 0.3090
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step
accuracy 0.9060693383216858
loss 0.33883601427078247
mcc 0.8745344072490733
auc_pr 0.8823978706009694
time predict 1.7318804264068604


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.9344 - loss: 0.2524
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step
accuracy 0.9248554706573486
loss 0.3017924726009369
mcc 0.8996514775424904
auc_pr 0.895048569992917
time predict 1.833791971206665


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.9178 - loss: 0.2735
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step
accuracy 0.913294792175293
loss 0.31921491026878357
mcc 0.8841380984832153
auc_pr 0.8913953073507553
time predict 1.8367140293121338


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [56]:
df_histories['predict_accuracies']=accuracies_predict
df_histories['predict_losses']=loss_predict
df_histories['times_predict']=times_predict
df_histories['mccs']=mccs_predict
df_histories['aucpr']=aucpr_predict
df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsGraph/early_best_finall_predict.csv')

In [57]:
df_histories["patience"]=np.linspace(5,70,14)

In [58]:
df_histories

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,predict_accuracies,predict_losses,times_predict,mccs,aucpr,patience
0,59.287467,0.975147,0.909747,1.840268,2.005214,1.000000e-07,0.903179,0.397968,1.648784,0.870638,0.871097,5.0
1,68.628585,0.990511,0.931408,1.914578,2.010741,1.000000e-05,0.921965,0.313010,1.609209,0.895784,0.887786,10.0
2,73.777139,0.990059,0.927798,1.864615,2.022250,1.000000e-08,0.906069,0.338836,1.731880,0.874534,0.882398,15.0
3,90.885048,0.998192,0.936823,1.925896,2.052175,1.000000e-05,0.929191,0.279116,1.594833,0.905440,0.893857,20.0
4,77.420167,0.996837,0.938628,1.918162,2.032023,1.000000e-06,0.921965,0.292859,1.698569,0.895762,0.888463,25.0
5,89.939603,0.998192,0.940433,1.926531,2.043819,1.000000e-05,0.926301,0.282845,1.644541,0.901724,0.890711,30.0
6,85.963678,0.998644,0.945848,1.903825,2.034818,1.000000e-06,0.932081,0.290505,1.698132,0.909249,0.891577,35.0
7,92.671117,0.998644,0.938628,1.866122,2.037306,1.000000e-05,0.917630,0.296122,1.675654,0.889941,0.890245,40.0
8,101.977762,0.999548,0.936823,1.940450,2.022266,1.000000e-06,0.920520,0.296401,1.685147,0.893845,0.891677,45.0
9,98.848299,0.999096,0.940433,1.873892,2.011722,1.000000e-06,0.920520,0.288502,3.348567,0.893864,0.896177,50.0


In [59]:
df_histories[(df_histories['aucpr']>0.889)]

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,predict_accuracies,predict_losses,times_predict,mccs,aucpr,patience
3,90.885048,0.998192,0.936823,1.925896,2.052175,0.000010,0.929191,0.279116,1.594833,0.905440,0.893857,20.0
5,89.939603,0.998192,0.940433,1.926531,2.043819,0.000010,0.926301,0.282845,1.644541,0.901724,0.890711,30.0
6,85.963678,0.998644,0.945848,1.903825,2.034818,0.000001,0.932081,0.290505,1.698132,0.909249,0.891577,35.0
7,92.671117,0.998644,0.938628,1.866122,2.037306,0.000010,0.917630,0.296122,1.675654,0.889941,0.890245,40.0
8,101.977762,0.999548,0.936823,1.940450,2.022266,0.000001,0.920520,0.296401,1.685147,0.893845,0.891677,45.0
9,98.848299,0.999096,0.940433,1.873892,2.011722,0.000001,0.920520,0.288502,3.348567,0.893864,0.896177,50.0
10,94.968059,0.999096,0.942238,1.918782,2.008884,0.000001,0.924855,0.288975,1.836705,0.899583,0.890958,55.0
11,89.313888,0.999548,0.940433,1.875087,2.028150,0.000100,0.913295,0.305293,1.782608,0.884339,0.892422,60.0
12,90.534475,0.999096,0.942238,1.923411,2.012146,0.000010,0.924855,0.301792,1.833792,0.899651,0.895049,65.0
13,96.701430,0.999548,0.944043,1.891084,2.016802,0.000100,0.913295,0.319215,1.836714,0.884138,0.891395,70.0


In [60]:
df_histories[(df_histories['aucpr']>0.889)&(df_histories['max_accuracies']-df_histories['max_val_accuracies']<0.055)]

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,predict_accuracies,predict_losses,times_predict,mccs,aucpr,patience
6,85.963678,0.998644,0.945848,1.903825,2.034818,0.000001,0.932081,0.290505,1.698132,0.909249,0.891577,35.0


Early stoping 40 the best